# MoXpert — Full Pipeline & Evaluation (Google Colab)

รวมทุกขั้นตอนสำหรับรันโมเดล MoXpert บน Colab ตามลำดับ ตั้งแต่ setup → build memory index → รัน evaluation → วัด accuracy เทียบกับ paper เดิม พร้อม confusion matrix

**ลำดับการทำงาน:**
1. เช็ค GPU / mount Drive / clone repo / ติดตั้ง dependencies
2. เตรียม dataset (แตกไฟล์ → symlink → เติมภาพ good ที่ขาด)
3. Build memory index (CLIP + FAISS)
4. รัน evaluation (Qwen2-VL-7B) → `Results_Qwen2VL.csv`
5. วิเคราะห์ผล: accuracy ต่อ question type + เทียบ paper + confusion matrix

> ก่อนรัน: Runtime → Change runtime type → เลือก **GPU** (แนะนำ A100/L4 บน Colab Pro)

## 1. เช็ค GPU

In [ ]:
!nvidia-smi
import torch
print(torch.cuda.is_available())

## 2. Mount Google Drive

เก็บไฟล์ archive ของ dataset (`.tar.gz`) และผลลัพธ์ไว้ที่ `MyDrive/MoXpert_data/`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone repo (branch `Original+SHAP`)

ต้องระบุ branch เพราะโค้ดที่แก้ไว้ทั้งหมดอยู่บน `Original+SHAP` ไม่ใช่ default branch

In [ ]:
# 1. ย้ายตำแหน่งกลับมาที่โฟลเดอร์หลักของ Colab
#%cd /content

# 2. ลบโฟลเดอร์ MoXpert เก่าออกให้สะอาด (ถ้ามีค้างอยู่)
#!rm -rf /content/MoXpert

!apt-get install -y git-lfs
!git lfs install

# 3. Clone branch Original มาใหม่
!git clone -b Original https://github.com/Poomdevkub/MoXpert.git /content/MoXpert

# 4. ย้ายเข้าโฟลเดอร์ MoXpert
%cd /content/MoXpert

# 5. เช็คความถูกต้อง
!git branch

## 4. ติดตั้ง dependencies

Colab session ใหม่ทุกครั้งต้องติดตั้งใหม่ (ไม่ persistent) ใช้เวลาสัก 1-2 นาที

In [ ]:
!pip install -r requirements.txt

## 5. แตกไฟล์ dataset ลง `/content`

แตกลง local disk ของ Colab (`/content`) ไม่ใช่อ่านตรงจาก Drive mount — เพราะ Drive FUSE อ่านไฟล์เล็กจำนวนมากช้ามาก

> ต้องอัปโหลด `MVTec-AD.tar.gz`, `VisA.tar.gz`, `DS-MVTec.tar.gz` ไว้ใน `MyDrive/MoXpert_data/` ก่อน (บีบอัดจาก Mac ด้วย `COPYFILE_DISABLE=1 tar --no-xattrs`)

In [ ]:
!mkdir -p /content/Dataset/MMAD
!tar xzf /content/drive/MyDrive/MoXpert_data/MVTec-AD.tar.gz -C /content/Dataset/MMAD --no-same-owner
!tar xzf /content/drive/MyDrive/MoXpert_data/VisA.tar.gz -C /content/Dataset/MMAD --no-same-owner
!tar xzf /content/drive/MyDrive/MoXpert_data/DS-MVTec.tar.gz -C /content/Dataset/MMAD --no-same-owner

In [ ]:
!find /content/Dataset/MMAD -name "*.png" | wc -l
!find /content/Dataset/MMAD/MVTec-AD -name "*.png" | wc -l   # ควรได้ 6612
!find /content/Dataset/MMAD/VisA -name "*.png" | wc -l        # ควรได้ 1200
!find /content/Dataset/MMAD/DS-MVTec -name "*.png" | wc -l    # ควรได้ 4090

In [ ]:
!ls -la /content/drive/MyDrive/MoXpert_data/DS-MVTec.tar.gz
!ls -la /content/Dataset/MMAD

## 6. Symlink dataset เข้ากับ repo

โค้ดอ้าง `../Dataset/MMAD/...` (= `/content/MoXpert/Dataset/MMAD`) ต้อง link ให้ตรงกับที่แตกไฟล์ไว้ที่ `/content/Dataset` ไม่งั้นหาไฟล์ไม่เจอ

In [ ]:
!ln -s /content/Dataset /content/MoXpert/Dataset
!ls -la /content/MoXpert/Dataset/MMAD

## 7. เติมภาพ "good" ที่ขาดใน DS-MVTec

DefectSpectrum เผยแพร่ภาพ good (ไม่มีตำหนิ) มาไม่ครบทุกไฟล์โดยตั้งใจ เพราะเป็นภาพชุดเดียวกับ `MVTec-AD/{category}/test/good/` — สคริปต์นี้ copy ไฟล์ที่ขาดมาเติมให้ (ไม่กระทบไฟล์เดิม)

In [ ]:
%cd /content/MoXpert/Experiments
!python fix_missing_good_images.py

In [ ]:
!find /content/Dataset/MMAD -name "*.png" | wc -l

## 8. Build Memory Index (CLIP + FAISS)

สร้าง FAISS index จากภาพ `train/` ของ MVTec-AD + VisA ผลลัพธ์: `Memory/memory.index` และ `Memory/reference_image_locations.txt`

In [ ]:
%cd /content/MoXpert/Memory
!python build_memory.py

## 9. รัน Evaluation (Qwen2-VL-7B)

ประเมินผลกับ DS-MVTec (1,691 ภาพ / 6,507 คำถาม) ผลลัพธ์บันทึกลง `Experiments/Results_Qwen2VL.csv` แบบเขียนทีละแถว

> **ใช้เวลานาน** (หลายชั่วโมงบน GPU เดียว) หากมี `Results_Qwen2VL.csv` อยู่แล้วและไม่อยากรันซ้ำ ให้ข้ามเซลล์นี้ไปที่ส่วนวิเคราะห์ได้เลย

In [ ]:
%cd /content/MoXpert/Experiments
!python Qwen2-VL.py

## 10. วิเคราะห์ผล — Accuracy ต่อ Question Type

อ่าน `Results_Qwen2VL.csv` แล้ว join กับ annotation เพื่อหา question type (CSV ไม่มีคอลัมน์ type) จากนั้นดึงตัวอักษรคำตอบ (A-D) จากทั้ง prediction และเฉลย มาคำนวณ accuracy

In [ ]:
import json, re
import pandas as pd

REPO = '/content/MoXpert'
CSV_PATH = f'{REPO}/Experiments/Results_Qwen2VL.csv'
ANN_PATH = f'{REPO}/Annotation/DS-MVTec.json'

# lookup: (image_path, question) -> question type
with open(ANN_PATH) as f:
    ann = json.load(f)
qtype = {}
for img, v in ann.items():
    for c in v['conversation']:
        qtype[(img, c['Question'])] = c['type']

def extract_letter(s):
    """ดึงตัวอักษรตัวเลือก A-D จากคำตอบ โดยไม่จับตัวอักษรที่อยู่กลางคำ"""
    if s is None:
        return None
    t = str(s).strip()
    if t == '' or t.upper() == 'N/A':
        return None
    t = t.upper()
    m = re.match(r'^([A-D])\b', t)      # ขึ้นต้นด้วยตัวเลือกเลย เช่น 'A', 'A.', 'A: Yes'
    if m:
        return m.group(1)
    m = re.search(r'\b([A-D])\b', t)   # ตัวเลือกยืนเดี่ยวกลางประโยค เช่น 'The answer is C'
    return m.group(1) if m else None

df = pd.read_csv(CSV_PATH)
df['Type'] = df.apply(lambda r: qtype.get((r['Image Path'], r['Question']), 'Unknown'), axis=1)
df['Pred'] = df['Predicted Answer'].apply(extract_letter)
df['True'] = df['Correct Answer'].apply(extract_letter)
df['Correct'] = df['Pred'] == df['True']

n_unparsed = df['Pred'].isna().sum()
print(f'Total rows: {len(df)}')
print(f'Unparseable predictions (นับเป็นผิด): {n_unparsed}')
print(f'Rows with Unknown type (join ไม่ติด): {(df.Type == "Unknown").sum()}')

In [ ]:
# accuracy ต่อ type + ภาพรวม
ORDER = ['Anomaly Detection', 'Defect Classification', 'Defect Localization',
         'Defect Description', 'Defect Analysis']

per_type = df.groupby('Type')['Correct'].agg(['mean', 'count'])
per_type = per_type.reindex([t for t in ORDER if t in per_type.index])
per_type['accuracy_%'] = (per_type['mean'] * 100).round(2)

micro = df['Correct'].mean() * 100                 # เฉลี่ยทุกคำถาม
macro = per_type['mean'].mean() * 100              # เฉลี่ยของ per-type

print(per_type[['accuracy_%', 'count']])
print()
print(f'Micro-average (ทุกคำถาม):     {micro:.2f}%')
print(f'Macro-average (เฉลี่ย 5 type): {macro:.2f}%')

## 11. เทียบกับ Paper เดิม

`PAPER_ACCURACY` กรอกค่าจาก paper **Table 1 (MVTec-AD)** แถว **Qwen2-VL (+MoXpert) 7B** ไว้แล้ว (DS-MVTec คือ defect subset ของ MVTec-AD จึงเทียบกับ Table 1) — มี baseline Qwen2-VL 7B เป็น comment เผื่อสลับ

> อ้างอิง: Chen & Imani, *A multi-expert framework for enhancing multimodal large language models in industrial anomaly detection*, Pattern Recognition 2025

In [ ]:
# ค่าจาก paper Table 1 (MVTec-AD) แถว Qwen2-VL (+MoXpert) 7B
# หมายเหตุ: paper เรียก 'Anomaly Discrimination' = 'Anomaly Detection' ในโค้ดนี้
PAPER_ACCURACY = {
    'Anomaly Detection':     89.65,
    'Defect Classification': 72.86,
    'Defect Localization':   76.11,
    'Defect Description':    85.57,
    'Defect Analysis':       93.20,
    'Average':               83.48,
}

# Baseline Qwen2-VL 7B (ไม่มี MoXpert) บน MVTec-AD เผื่ออยากเทียบ — สลับมาใช้ได้:
# PAPER_ACCURACY = {'Anomaly Detection': 82.26, 'Defect Classification': 68.46,
#     'Defect Localization': 76.11, 'Defect Description': 82.19,
#     'Defect Analysis': 92.28, 'Average': 80.26}

rows = []
for t in ORDER:
    if t not in per_type.index:
        continue
    ours = per_type.loc[t, 'accuracy_%']
    paper = PAPER_ACCURACY.get(t)
    diff = round(ours - paper, 2) if paper is not None else None
    rows.append({'Question Type': t, 'Ours (%)': ours,
                 'Paper (%)': paper if paper is not None else '—',
                 'Diff': diff if diff is not None else '—'})

paper_avg = PAPER_ACCURACY.get('Average')
rows.append({'Question Type': 'Average (macro)', 'Ours (%)': round(macro, 2),
             'Paper (%)': paper_avg if paper_avg is not None else '—',
             'Diff': round(macro - paper_avg, 2) if paper_avg is not None else '—'})

cmp_df = pd.DataFrame(rows)
cmp_df

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

types = [r['Question Type'] for r in rows]
ours_vals = [r['Ours (%)'] for r in rows]

def paper_of(t):
    v = PAPER_ACCURACY.get('Average') if 'Average' in t else PAPER_ACCURACY.get(t)
    return v if isinstance(v, (int, float)) else np.nan
paper_vals = [paper_of(t) for t in types]

x = np.arange(len(types))
w = 0.38
fig, ax = plt.subplots(figsize=(11, 5))
ax.bar(x - w/2, ours_vals, w, label='Ours', color='#4C78A8')
ax.bar(x + w/2, paper_vals, w, label='Paper', color='#F58518')
ax.set_ylabel('Accuracy (%)')
ax.set_title('MoXpert — Accuracy by Question Type (Ours vs Paper)')
ax.set_xticks(x)
ax.set_xticklabels(types, rotation=20, ha='right')
ax.legend()
ax.grid(axis='y', alpha=0.3)
for i, v in enumerate(ours_vals):
    ax.text(x[i] - w/2, v + 0.5, f'{v:.1f}', ha='center', fontsize=8)
for i, v in enumerate(paper_vals):
    if not np.isnan(v):
        ax.text(x[i] + w/2, v + 0.5, f'{v:.1f}', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 12. Confusion Matrix ต่อ Question Type

แสดง confusion matrix (แถว = เฉลยจริง, คอลัมน์ = คำตอบโมเดล) แยกตามแต่ละ question type — Anomaly Detection เป็น 2 ตัวเลือก (A/B) ที่เหลือเป็น 4 ตัวเลือก (A-D)

In [ ]:
from sklearn.metrics import confusion_matrix

present = [t for t in ORDER if t in df['Type'].unique()]
n = len(present)
fig, axes = plt.subplots(1, n, figsize=(4.2 * n, 4))
if n == 1:
    axes = [axes]

for ax, t in zip(axes, present):
    sub = df[df['Type'] == t].copy()
    labels = sorted(set(sub['True'].dropna()) | set(sub['Pred'].dropna()))
    # แทน prediction ที่ parse ไม่ได้ด้วย '?' เพื่อให้เห็นในเมทริกซ์
    y_true = sub['True'].fillna('?')
    y_pred = sub['Pred'].fillna('?')
    all_labels = sorted(set(y_true) | set(y_pred))
    cm = confusion_matrix(y_true, y_pred, labels=all_labels)
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(all_labels)))
    ax.set_yticks(range(len(all_labels)))
    ax.set_xticklabels(all_labels)
    ax.set_yticklabels(all_labels)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    acc = (sub['Correct'].mean() * 100)
    ax.set_title(f'{t}\n(acc {acc:.1f}%)', fontsize=10)
    for i in range(len(all_labels)):
        for j in range(len(all_labels)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black', fontsize=8)
plt.tight_layout()
plt.show()

## 13. เก็บผลลัพธ์กลับ Google Drive

`/content` หายเมื่อ session จบ — copy ผลลัพธ์และไฟล์วิเคราะห์กลับ Drive ก่อนปิด

In [ ]:
DRIVE = '/content/drive/MyDrive/MoXpert_data'
# บันทึกตารางเทียบ + accuracy ต่อ type
per_type.to_csv(f'{DRIVE}/accuracy_by_type.csv')
cmp_df.to_csv(f'{DRIVE}/accuracy_vs_paper.csv', index=False)
!cp /content/MoXpert/Experiments/Results_Qwen2VL.csv {DRIVE}/
!cp /content/MoXpert/Memory/memory.index {DRIVE}/ 2>/dev/null || true
print('Saved to', DRIVE)